## Avaliação do algoritmo Thompson Sampling com golden set

O objetivo desse notebook é validar se o algoritmo Contextual Thompson Sampling realiza a recomendação ideal de acordo com o contexto do cliente. Para isso foram selecionados 5 clientes com contextos específicos formando o golden set de avaliação.

### Critério de aprovação

Cada caso de teste contém:
- o contexto do cliente (idade, cargo, escolaridade, resultado da campanha anterior e número de contatos prévios);
- a oferta esperada (expected_arm);
- a justificativa da recomendação.

Ao final da execução, a recomendação produzida pelo algoritmo (recommended_arm) é comparada com a recomendação esperada, classificando cada caso como **PASS** ou **FAIL**.

In [2]:
import pandas as pd
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.policies.thompson_sampling import ThompsonSampling 

In [ ]:
# treinamento Thompson utilizando histórico sintético
offer_events = pd.read_csv(
    "../data/synthetic_enrichment/offer_events.sample.csv",
    parse_dates=["event_date"]
)

thompson = ThompsonSampling(random_state=42)

thompson.run(offer_events)

,event_id,customer_id,event_date,chosen_arm,reward_probability,reward,policy
0,1,1,2026-01-08,3,0.10,0,Contextual Thompson Sampling
1,2,1,2026-01-26,2,0.25,0,Contextual Thompson Sampling
2,3,1,2026-01-11,4,0.10,0,Contextual Thompson Sampling
3,4,1,2026-01-08,3,0.10,0,Contextual Thompson Sampling
4,5,2,2026-01-24,1,0.10,0,Contextual Thompson Sampling
...,...,...,...,...,...,...,...
123283,123284,41175,2026-01-15,1,0.20,0,Contextual Thompson Sampling
123284,123285,41175,2026-01-03,1,0.20,0,Contextual Thompson Sampling
123285,123286,41175,2026-01-12,1,0.20,0,Contextual Thompson Sampling
123286,123287,41176,2026-01-20,1,0.20,0,Contextual Thompson Sampling


In [ ]:
# leitura e transformação do golden set
golden_set = pd.read_json("../data/golden_set/evaluation_cases.json")

context = pd.json_normalize(golden_set["context"])

golden_set = pd.concat(
    [golden_set.drop(columns="context"), context],
    axis=1
)

golden_set

,case_id,expected_arm,justification,age,job,education,poutcome,previous
0,1,1,Clientes com ensino superior e cargo de gestão...,43,management,university.degree,success,2
1,2,3,Clientes jovens tendem a responder melhor a um...,22,student,high.school,nonexistent,0
2,3,2,Clientes com ensino básico tendem a responder ...,36,services,basic.9y,failure,1
3,4,4,Clientes com histórico recente de sucesso em c...,27,technician,university.degree,success,3
4,5,2,Cliente sem histórico de campanhas anteriores ...,25,technician,high.school,nonexistent,0


In [ ]:
# o modelo Thompson recebe os casos do golden set
recommended = []

for _, customer in golden_set.iterrows():
    recommended.append(
        thompson.recommend_arm(customer)
    )

golden_set["recommended_arm"] = recommended

In [ ]:
# avaliação do resultado a partir do braço esperado 
golden_set["result"] = (
    golden_set["recommended_arm"]
    == golden_set["expected_arm"]
)

golden_set["result"] = golden_set["result"].map(
    {
        True: "PASS",
        False: "FAIL"
    }
)

In [ ]:
golden_set

,case_id,expected_arm,justification,age,job,education,poutcome,previous,recommended_arm,result
0,1,1,Clientes com ensino superior e cargo de gestão...,43,management,university.degree,success,2,4,FAIL
1,2,3,Clientes jovens tendem a responder melhor a um...,22,student,high.school,nonexistent,0,3,PASS
2,3,2,Clientes com ensino básico tendem a responder ...,36,services,basic.9y,failure,1,2,PASS
3,4,4,Clientes com histórico recente de sucesso em c...,27,technician,university.degree,success,3,4,PASS
4,5,2,Cliente sem histórico de campanhas anteriores ...,25,technician,high.school,nonexistent,0,2,PASS
